# 03 — Spatiotemporal Feature Engineering

**Notebook responsibility:** temporal features, spatial features, documented crime-category mapping, daily aggregation to the community_area × date modeling unit, and historical lag/rolling features (Phases 4, 5, 11 — combined here because 11 operates directly on 5's output). Notebook orchestrates; logic lives in `src/features/`.

**Aim:** turn `data/interim/crime_cleaned.parquet` (incident-level) into `data/interim/crime_daily.parquet` (community_area × date, with lag/rolling history) — the frame `04_crime_eda`, `06_crime_weather_analysis`, and `07_model_dataset_creation` all build on.

**Leakage discipline:** every lag/rolling feature is computed from strictly prior days only (verified explicitly in Section 6). This is the modeling unit's single most important correctness property — get it wrong here and every downstream metric is invalid.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import DATA_INTERIM
from src.features.temporal_features import create_temporal_features
from src.features.spatial_features import add_crime_category, create_spatial_features, aggregate_daily_crime, CRIME_CATEGORY_MAP
from src.features.crime_lags import create_crime_lag_features, LAG_DAYS, ROLLING_WINDOWS

pd.set_option("display.max_columns", 60)

_Findings: (fill in after running)_

## 1. Load Cleaned Crime Data

**Aim:** load `02_crime_cleaning`'s output. Incident-level, one row per crime record.

In [ ]:
crime = pd.read_parquet(DATA_INTERIM / "crime_cleaned.parquet")
print(f"Shape: {crime.shape}")
crime[["date", "community_area", "primary_type", "arrest", "domestic"]].head(3)

_Findings: (confirm shape matches notebook-02 output; dtypes already correct from cleaning)_

## 2. Temporal Features

**Aim:** derive year/month/day/day_of_week/week_of_year/quarter/season/is_weekend/hour/time_of_day via `create_temporal_features()`, then validate every derived field against its expected domain — a feature-engineering bug here silently corrupts every downstream aggregation.

In [ ]:
crime = create_temporal_features(crime)
crime[["date", "year", "month", "day_of_week", "week_of_year", "quarter", "season", "is_weekend", "hour", "time_of_day"]].head(3)

_Findings: (spot-check a few rows by hand — does `season`/`time_of_day` match the `date`/`hour` shown?)_

### 2.1 Domain validation

**Aim:** every derived field must fall inside its known valid range, and `is_weekend` must agree with `day_of_week` by construction — assert rather than eyeball.

In [ ]:
assert crime["year"].between(2021, 2025).all(), "year outside configured scope"
assert crime["month"].between(1, 12).all()
assert crime["day_of_week"].between(0, 6).all()
assert crime["hour"].between(0, 23).all()
assert (crime["is_weekend"] == crime["day_of_week"].isin([5, 6])).all()
assert crime["season"].isin(["Winter", "Spring", "Summer", "Fall"]).all()
assert crime["time_of_day"].isin(["Night", "Morning", "Afternoon", "Evening"]).all()
print("All domain checks passed.")
print()
print("Day-of-week distribution:")
print(crime["day_of_week"].value_counts().sort_index())
print()
print("Season distribution:")
print(crime["season"].value_counts())

_Findings: (assertions pass; is the day-of-week distribution roughly even, or does a specific day stand out — worth flagging for `04_crime_eda`? Does season distribution look plausible for Chicago's seasonal crime pattern?)_

## 3. Crime Category Mapping

**Aim:** map `primary_type` to a documented broad category via `add_crime_category()` — five groups (Violent/Property/Drug/Fraud/Other), explicit mapping in `CRIME_CATEGORY_MAP`, no undocumented logic. Check coverage: a high-volume `primary_type` landing in `OTHER` means the map needs expanding, not that the data is wrong.

In [ ]:
crime = add_crime_category(crime)

category_counts = crime["crime_category"].value_counts()
print(category_counts)
print(f"\nOTHER share: {category_counts.get('OTHER', 0) / len(crime):.2%}")

print("\nTop primary_types currently falling into OTHER:")
print(crime.loc[crime["crime_category"] == "OTHER", "primary_type"].value_counts().head(10))

_Findings: (fill in after running — is OTHER's share reasonable, or does a high-volume type need adding to `CRIME_CATEGORY_MAP` in `src/features/spatial_features.py`? Update the map, not this notebook, if so — the map is the documented source of truth.)_

## 4. Spatial Feature Flag

**Aim:** flag coordinate availability via `create_spatial_features()`. No new geometry is derived — cross-check against notebook 01/02's known ~1.45% coordinate-null rate.

In [ ]:
crime = create_spatial_features(crime)
coord_null_rate = 1 - crime["has_coordinates"].mean()
print(f"Coordinate-null rate: {coord_null_rate:.2%}")

_Findings: (should match ~1.45% from notebooks 01/02; a material difference means something changed upstream — investigate before proceeding)_

## 5. Daily Aggregation

**Aim:** collapse incident-level rows to the community_area × date modeling unit via `aggregate_daily_crime()`, reindexed to the full date range × every community area so zero-crime days are explicit rows, not gaps.

In [ ]:
daily = aggregate_daily_crime(crime)
print(f"Shape: {daily.shape}")
daily.head(3)

### 5.1 Aggregation correctness checks

**Aim:** two hard invariants must hold or the aggregation has a bug: (1) shape must equal `n_community_areas × n_days` exactly — every combination present, none missing, none duplicated; (2) the sum of `crime_count` across the daily frame must equal the incident-level row count — every record counted exactly once, none dropped or double-counted.

In [ ]:
n_areas = crime["community_area"].nunique()
n_days = (crime["date"].dt.normalize().max() - crime["date"].dt.normalize().min()).days + 1
expected_rows = n_areas * n_days

assert len(daily) == expected_rows, f"expected {expected_rows} rows ({n_areas} areas x {n_days} days), got {len(daily)}"
assert daily["crime_count"].sum() == len(crime), "daily crime_count sum does not match incident-level row count -- rows lost or double-counted"
assert (daily["violent_crime_count"] + daily["property_crime_count"] + daily["drug_crime_count"] <= daily["crime_count"]).all(), \
    "category subcounts exceed total crime_count -- a row was counted into more than one category"

print(f"Shape check passed: {len(daily):,} rows ({n_areas} areas x {n_days} days).")
print(f"Conservation check passed: daily crime_count sums to {daily['crime_count'].sum():,}, matching {len(crime):,} incident rows.")
print(f"Category-subcount check passed: violent + property + drug never exceeds crime_count per row.")

_Findings: (both checks pass by construction if `aggregate_daily_crime` is correct — record the actual area/day counts here for reference in later notebooks)_

## 6. Historical Crime Lag/Rolling Features

**Aim:** add `crime_count_lag_{1,3,7,14,28}`, `crime_rolling_{7,14,28}`, `recent_7d_avg`, `previous_7d_avg`, `trend_ratio`, `violent_ratio`, `property_ratio` via `create_crime_lag_features()`.

In [ ]:
daily = create_crime_lag_features(daily)
lag_cols = [f"crime_count_lag_{l}" for l in LAG_DAYS] + [f"crime_rolling_{w}" for w in ROLLING_WINDOWS]
daily[["community_area", "date", "crime_count"] + lag_cols].head(10)

_Findings: (fill in after running — for the first rows of any single community_area, all lag/rolling columns should be `NaN`; that's expected warm-up, not a defect)_

### 6.1 Leakage audit (mandatory)

**Aim:** prove — not assume — that every lag value is computed from strictly prior days. Spot-check: for every row, `crime_count_lag_1` on day D must equal `crime_count` on day D-1, within the same `community_area`. Also confirm the expected NaN warm-up length per lag.

In [ ]:
check = daily[["community_area", "date", "crime_count", "crime_count_lag_1"]].copy()
check["prior_day"] = check["date"] - pd.Timedelta(days=1)

prior_lookup = daily.set_index(["community_area", "date"])["crime_count"]
check["expected_lag_1"] = [
    prior_lookup.get((ca, pd_), pd.NA) for ca, pd_ in zip(check["community_area"], check["prior_day"])
]

mismatch = check.dropna(subset=["crime_count_lag_1"])
mismatch = mismatch[mismatch["crime_count_lag_1"] != mismatch["expected_lag_1"]]
assert len(mismatch) == 0, f"{len(mismatch)} rows where crime_count_lag_1 does not equal the prior day's crime_count"

print("Leakage audit passed: crime_count_lag_1 exactly equals crime_count from the prior calendar day, every row, every community area.")
print()
for lag in LAG_DAYS:
    n_nan = daily[f"crime_count_lag_{lag}"].isna().sum()
    expected_nan = lag * n_areas
    print(f"crime_count_lag_{lag}: {n_nan:,} NaN (expected {expected_nan:,} = {lag} warm-up days x {n_areas} areas)")

_Findings: (leakage audit passed — this is the load-bearing check for the entire modeling phase; confirm NaN counts match the formula exactly, not approximately)_

## 7. Save Engineered Daily Dataset

**Aim:** persist to `data/interim/crime_daily.parquet` — the input for `04_crime_eda`, `06_crime_weather_analysis`, and `07_model_dataset_creation`. Warm-up NaNs are saved as-is; `07` handles them explicitly during the chronological split, not here.

In [ ]:
out_path = DATA_INTERIM / "crime_daily.parquet"
daily.to_parquet(out_path, index=False)
print(f"Saved {len(daily):,} rows, {len(daily.columns)} columns to: {out_path}")

_Findings: (confirm file written, shape matches Section 5.1's expected_rows)_

## 8. Summary

**Top findings:**
- _(fill in after running)_

**Quality issues to address downstream:**
- Warm-up NaNs (28 rows per community area) in lag/rolling columns must be excluded from — or explicitly handled by — the chronological split in `07_model_dataset_creation`; they are not missing data to impute.
- `trend_ratio`/`violent_ratio`/`property_ratio` are `NaN` wherever their denominator (`previous_7d_avg` / `crime_rolling_7`) is 0 — zero-crime history, not a computation error. Confirm `08_model_training_and_comparison` handles this (e.g. via `class_weight` / imputation choice), not silent row-dropping.
- Any `primary_type` flagged in Section 3 as high-volume `OTHER` needs `CRIME_CATEGORY_MAP` expanded before `04_crime_eda`'s compositional analysis.

**Next step:** proceed to `04_crime_eda.ipynb`.